In [137]:
import pathlib
import os

os.chdir(pathlib.Path().absolute() / "..")
os.getcwd()

'c:\\Users\\connor\\programming'

In [138]:
%pip install positional-encodings[pytorch]

Note: you may need to restart the kernel to use updated packages.


In [139]:
from data import build_dataset
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from einops.layers.torch import Rearrange
from einops import rearrange
import torch
import math
import numpy as np
from positional_encodings.torch_encodings import PositionalEncoding3D, Summer, PositionalEncoding2D

In [140]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [141]:
raw = build_dataset(transform=None)
loader = DataLoader(raw, batch_size=1, shuffle=True)

FileNotFoundError: [Errno 2] No such file or directory: 'data.csv'

In [ ]:
class Attention(nn.Module):
    """
    This is much like `.vision_transformer.Attention` but uses *localised* self attention by accepting an input with
     an extra "image block" dim
    """

    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim**-0.5

        self.qkv = nn.Linear(dim, 3 * dim, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        """
        x is shape: B (batch_size), T (image blocks), N (seq length per image block), C (embed dim)
        """
        b, t, n, c = x.shape
        # result of next line is (qkv, B, num (H)eads, T, N, (C')hannels per head)
        qkv = self.qkv(x).reshape(b, t, n, 3, self.num_heads, c // self.num_heads).permute(3, 0, 4, 1, 2, 5)
        q, k, v = qkv.unbind(0)  # make torchscript happy (cannot use tensor as tuple)

        attn = (q @ k.transpose(-2, -1)) * self.scale  # (B, H, T, N, N)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).permute(0, 2, 3, 4, 1).reshape(b, t, n, c)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x  # (B, T, N, C)


class TransformerLayer(nn.Module):
    """
    This is much like `.vision_transformer.Block` but:
        - Called TransformerLayer here to allow for "block" as defined in the paper ("non-overlapping image blocks")
        - Uses modified Attention layer that handles the "block" dimension
    """

    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.0,
        qkv_bias=False,
        drop=0.0,
        attn_drop=0.0,
        drop_path=0.0,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm,
    ):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = Attention(dim, num_heads=num_heads, qkv_bias=qkv_bias, attn_drop=attn_drop, proj_drop=drop)
        # self.drop_path = DropPath(drop_path) if drop_path > 0.0 else nn.Identity()
        self.drop_path = nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        # self.mlp = Mlp(in_features=dim, hidden_features=mlp_hidden_dim, act_layer=act_layer, drop=drop)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden_dim),
            act_layer(),
            nn.Dropout(drop),
            nn.Linear(mlp_hidden_dim, dim),
            nn.Dropout(drop)
        )

    def forward(self, x):
        y = self.norm1(x)
        x = x + self.drop_path(self.attn(y))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

In [ ]:
def blockify(embeddings, block_size):
    """
    Given an array of embeddings with shape (batch_size, sequence_length, embedding_size), 
    blockify the embeddings into blocks with the given block size.
    
    Args:
        embeddings (ndarray): Array of embeddings with shape (batch_size, sequence_length, embedding_size).
        block_size (tuple): The desired block size, in the form (num_blocks, block_length, embedding_size).
        
    Returns:
        An array of blockified embeddings with shape (batch_size, num_blocks, block_length, embedding_size).
    """
    batch_size, seq_length, emb_size = embeddings.shape
    num_blocks, block_length, _ = block_size
    assert seq_length % block_length == 0, "Block length must evenly divide sequence length."
    blockified = np.reshape(embeddings, (batch_size, num_blocks, block_length, emb_size))
    return blockified

def deblockify(x, block_size: int):
    """blocks to image
    Args:
        x (Tensor): with shape (B, T, N, C) where T is number of blocks and N is sequence size per block
        block_size (int): edge length of a single square block in units of desired D, H, W
    """
    b, t, _, c = x.shape
    grid_size = round(math.pow(t, 1 / 3))
    depth = height = width = grid_size * block_size
    x = x.reshape(b, grid_size, grid_size, grid_size, block_size, block_size, block_size, c)

    x = x.permute(0, 1, 4, 2, 5, 3, 6, 7).reshape(b, depth, height, width, c)

    return x  # (B, D, H, W, C)

def _blockify(patch_projected, block_size):
    B, num_patches, emb_dim = patch_projected.shape
    num_blocks = num_patches // block_size
    blockified = patch_projected.view(B, num_blocks, block_size, emb_dim)
    return blockified

def unblock(x, level):
    B, T, n, C = x.shape
    block_size = int(round(np.cbrt(n)))
    blocks_per_dim = int(round(np.cbrt(T)))
    H = W = D = block_size * blocks_per_dim
    x = x.reshape(B, C, H, W, D)

    return x

In [ ]:
class Hierarchical(nn.Module):
    def __init__(self, embed_dim, num_layers, num_heads):
        super().__init__()
        # self.transformer_layers = nn.ModuleList([nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads) for _ in range(num_layers)])
        self.transformer_layers = nn.ModuleList([TransformerLayer(embed_dim, num_heads) for _ in range(num_layers)])   
    def forward(self, x):
        for layer in self.transformer_layers:
            x = layer(x)
        # x = self.block_aggregation(x)
        return x

In [ ]:
class Hierarchy(nn.Module):
    def __init__(self, patch_size, embed_dim=1024) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        self.conv = nn.LazyConv3d(embed_dim, 1).to(device)
        
    def forward(self, x):
        # Split input schematic into subvolumes

        pool = nn.MaxPool3d(2).to(device)
        # x = rearrange(x, "b t sh sw sd c -> b t (sh sw sd) c")
        x = pool(x)
        print("2) Pooling: ",x.shape)

        B, C, D, H, W = x.shape

        # Blockify
        x = (
            x.unfold(2, self.patch_size, self.patch_size)
            .unfold(3, self.patch_size, self.patch_size)
            .unfold(4, self.patch_size, self.patch_size)
        )
        
        print("3) Unfolded: ", x.shape)
        
        # Transformer
        
        x = rearrange(x, "b c p1 p2 p3 s1 s2 s3 -> b (p1 p2 p3) (s1 s2 s3) c")
        positional = Summer(PositionalEncoding2D(C))
        x = positional(x)
        
        # x = rearrange(x, "b t n c -> b t (n c)")
        print("4) Flatten for transformer: ", x.shape)
        

        # Transformer
        layer = Hierarchical(x.shape[-1], 2, 8).to(device)
        x = layer(x)

        print("5) Transformer output: ", x.shape)

        # Unblock
        # x = x.view(B, x.shape[1], self.patch_size, self.patch_size, self.patch_size, C)
        x = x.reshape(B, C, D, H, W)
        print("6) Unblocked: ", x.shape)
        
        x = self.conv(x)
        print("7) Conv: ", x.shape)

        return x

class Model(nn.Module):
    def __init__(self, patch_size=None, embed_dim=None, num_hierarchies=2):
        super().__init__()

        # fuck it
        if not embed_dim:
            embed_dim = [128, 256, 512]
        if not patch_size:
            patch_size = [4, 8, 16]
        
        self.embed_dim = embed_dim
        self.patch_size = patch_size
        self.proj = nn.LazyLinear(embed_dim[0]).to(device)

        self.nests = nn.ModuleList(
            [Hierarchy(p, e) for p, e in zip(patch_size, embed_dim)]
        )
        

    def forward(self, x):
        
        B, C, D, H, W = x.shape
        print("1) Projecting patches")

        proj_size = self.patch_size[0]
        patch_length = D // proj_size

        patches = (
            x.unfold(2, proj_size, proj_size)
            .unfold(3, proj_size, proj_size)
            .unfold(4, proj_size, proj_size)
        )

        # print(
        #     "Extraction: ", patches.shape
        # )  # (B, num_patches, patch_size, patch_size, patch_size, C)

        patches = rearrange(patches, "b c p1 p2 p3 s1 s2 s3 -> b p1 p2 p3 (s1 s2 s3 c)")
        # print("Flattening: ", patches.shape)
        patch_embeddings = self.proj(patches)
        # print("Projection: ", patch_embeddings.shape)
        
        positional = Summer(PositionalEncoding3D(self.embed_dim[0]))
        x = positional(patch_embeddings)
        x = rearrange(x, "b d h w c -> b c d h w")
        print("Patch embeddings: ", x.shape)
        
        encoder_outputs = []
        for nest in self.nests:
            print(f"===\nInput shape: {x.shape}")
            x = nest(x)
            encoder_outputs.append(x)
            
        return encoder_outputs

SIZE = 8
BATCH_SIZE = 2
loader = DataLoader(raw, batch_size=BATCH_SIZE, shuffle=True)
SHAPE = (BATCH_SIZE, 1, 128, 128, 128)
patch_size = 4

torch.cuda.empty_cache()

for x, y in loader:
    x = x.unsqueeze(1).to(device)
    # _x = x.clone()
    x = Model(patch_size=[4, 4, 4], embed_dim=[128, 256, 512])(x)

    # print(x.shape)
    # print(x[0, 0, :10])

    break


c:\Users\connor\miniconda3\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


1) Projecting patches
Patch embeddings:  torch.Size([2, 128, 32, 32, 32])
===
Input shape: torch.Size([2, 128, 32, 32, 32])
2) Pooling:  torch.Size([2, 128, 16, 16, 16])
3) Unfolded:  torch.Size([2, 128, 4, 4, 4, 4, 4, 4])
4) Flatten for transformer:  torch.Size([2, 64, 64, 128])
5) Transformer output:  torch.Size([2, 64, 64, 128])
6) Unblocked:  torch.Size([2, 128, 16, 16, 16])
7) Conv:  torch.Size([2, 128, 16, 16, 16])
===
Input shape: torch.Size([2, 128, 16, 16, 16])
2) Pooling:  torch.Size([2, 128, 8, 8, 8])
3) Unfolded:  torch.Size([2, 128, 2, 2, 2, 4, 4, 4])
4) Flatten for transformer:  torch.Size([2, 8, 64, 128])
5) Transformer output:  torch.Size([2, 8, 64, 128])
6) Unblocked:  torch.Size([2, 128, 8, 8, 8])
7) Conv:  torch.Size([2, 256, 8, 8, 8])
===
Input shape: torch.Size([2, 256, 8, 8, 8])
2) Pooling:  torch.Size([2, 256, 4, 4, 4])
3) Unfolded:  torch.Size([2, 256, 1, 1, 1, 4, 4, 4])
4) Flatten for transformer:  torch.Size([2, 1, 64, 256])
5) Transformer output:  torch.Size(

In [142]:
[i.shape for i in x]

[torch.Size([2, 128, 16, 16, 16]),
 torch.Size([2, 256, 8, 8, 8]),
 torch.Size([2, 512, 4, 4, 4])]

In [143]:
i = x[-1]
i.shape

torch.Size([2, 512, 4, 4, 4])

In [144]:
conv = nn.Sequential(
    nn.LayerNorm(i.shape[1:]),
    nn.Conv3d(i.shape[1], i.shape[1]*2, 4),
).to(device)
j = conv(i)
j.shape

torch.Size([2, 1024, 1, 1, 1])

In [146]:
upsc = nn.Sequential(
    nn.LazyConvTranspose3d(512, 4)
).to(device)
upsc(j).shape

torch.Size([2, 512, 4, 4, 4])

In [162]:
upsc = nn.Sequential(
    nn.LazyConvTranspose3d(128, 9)
).to(device)
upsc(torch.randn(2, 256, 8, 8, 8).to(device)).shape

torch.Size([2, 128, 16, 16, 16])

In [134]:
res = nn.Sequential(
    nn.LazyConv3d(256, 3),
    nn.LazyConv3d()
).to(device)

res(j).shape

c:\Users\connor\miniconda3\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


torch.Size([2, 3, 2, 2, 2])

In [10]:
i = torch.randn(2, 512, 8192)
i.view(2, 128, 32, 32, 32).shape

torch.Size([2, 128, 32, 32, 32])

In [11]:
conv = nn.Sequential(
    nn.Conv3d(256, 512, 1),
    nn.ConvTranspose3d(512, 128, 16)
).to(device)
conv(x_norm).shape

NameError: name 'x_norm' is not defined

In [ ]:
import torch
import torch.nn as nn

# Define input volume (B, C, H, W, D)
input_volume = torch.randn(2, 1, 128, 128, 128)

# Tokenize the input volume
num_bins = 64
tokenized_volume = torch.bucketize(input_volume, torch.linspace(input_volume.min(), input_volume.max(), num_bins + 1))
print(tokenized_volume.shape)

# Flatten channel and spatial dimensions (B, N)
B, C, H, W, D = tokenized_volume.shape
flattened_tokens = tokenized_volume.view(B, C * H * W * D)
print(flattened_tokens.shape)

# Embed the tokens (B, N, E)
embedding_size = 16
embedding = nn.Embedding(num_bins + 1, embedding_size)
embedded_tokens = embedding(flattened_tokens)
print(embedded_tokens.shape)

# Reshape back to original spatial dimensions (B, C, H, W, D, E)
embedded_volume = embedded_tokens.view(B, C, H, W, D, embedding_size)
print(embedded_volume.shape)

# Subdivide the volume into 8x8x8 cubes
subdivision_size = 8
subdivided_volume = embedded_volume.unfold(2, subdivision_size, subdivision_size).unfold(3, subdivision_size, subdivision_size).unfold(4, subdivision_size, subdivision_size)

# Permute to get shape (B, C, S, S, S, 8, 8, 8, E)
_, _, sh, sw, sd, _, _, _, _ = subdivided_volume.shape
subdivided_volume = subdivided_volume.permute(0, 1, 2, 3, 4, 7, 5, 6, 8).contiguous().view(B, C, sh * sw * sd, subdivision_size, subdivision_size, subdivision_size, embedding_size)

# Process the subdivided volume with a transformer
d_model = embedding_size
nhead = 4
num_layers = 2
transformer_layer = nn.TransformerEncoderLayer(d_model, nhead)
transformer = nn.TransformerEncoder(transformer_layer, num_layers=num_layers)
transformer_output = transformer(subdivided_volume.view(B, sh * sw * sd, -1))

# Reshape back to (B, C, S, S, S, E)
local_transformer_output = transformer_output.view(B, C, sh, sw, sd, -1)


torch.Size([2, 1, 128, 128, 128])
torch.Size([2, 2097152])
torch.Size([2, 2097152, 16])
torch.Size([2, 1, 128, 128, 128, 16])


AssertionError: was expecting embedding dimension of 16, but got 8192

In [ ]:
i = torch.randn(2, 64, 8, 128)
transformer = TransformerLayer(64, 8, 4)
o = transformer(i)
print(o.shape)

AssertionError: was expecting embedding dimension of 64, but got 128

In [ ]:

# Input
batch_size = 2
channels = 1
height = 128
width = 128
depth = 128
input_shape = (batch_size, channels, height, width, depth)
input_volume = torch.randn(input_shape)

# Patch projection
patch_size = 8
embed_dim = 256

class PatchProjection(nn.Module):
    def __init__(self, in_channels, out_channels, patch_size):
        super().__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        return self.conv(x)

proj = PatchProjection(channels, embed_dim, patch_size)
projected_patches = proj(input_volume)

# Intuition: Input volumes are divided into non-overlapping patches and projected onto a higher-dimensional space
projected_patches.shape

torch.Size([2, 256, 16, 16, 16])

In [ ]:
# Blockify
block_size = 64
num_blocks = 64
sequence_length = 64

def blockify(x, block_size):
    b, c, h, w, d = x.shape
    num_blocks = (h * w * d) // block_size
    x = x.permute(0, 2, 3, 4, 1).contiguous()
    x = x.view(b, h * w * d, c)
    x = x.view(b, num_blocks, block_size, c)
    return x

blockified_patches = blockify(projected_patches, block_size)

# Intuition: Projected patches are rearranged into blocks for efficient non-local communication
blockified_patches.shape

torch.Size([2, 64, 64, 256])

In [ ]:
# Transformer layer
from torch.nn import TransformerEncoderLayer

d_model = 256
nhead = 8
num_layers = 1

transformer_layer = TransformerEncoderLayer(d_model, nhead)
transformer = nn.TransformerEncoder(transformer_layer, num_layers)

transformed_patches = transformer(blockified_patches)

# Intuition: The transformer layer captures relationships between patch embeddings and learns hierarchical representations
transformed_patches.shape

AssertionError: query should be unbatched 2D or batched 3D tensor but received 4-D query tensor

In [ ]:
import torch
import torch.nn as nn

# Input volume shape: [batch size, num channels, depth, height, width]
x = torch.randn(2, 3, 128, 128, 128)

# Patch projection
patch_size = (16, 16, 16)
stride = patch_size
projection = nn.Conv3d(in_channels=3, out_channels=128, kernel_size=patch_size, stride=stride)
x_patches = projection(x)

# Blockify
n_blocks = (int(x_patches.shape[-3] / 4), int(x_patches.shape[-2] / 4), int(x_patches.shape[-1] / 4))
x_blocks = x_patches.unfold(-3, 4, 4).unfold(-2, 4, 4).unfold(-1, 4, 4)
x_blocks = x_blocks.reshape(x_blocks.shape[0], -1, x_blocks.shape[-4], x_blocks.shape[-3], x_blocks.shape[-2], x_blocks.shape[-1])

# Print shapes
print("Input shape:", x.shape)  # [2, 3, 128, 128, 128]
print("Patch projection shape:", x_patches.shape)  # [2, 128, 8, 8, 8]
print("Blockify shape:", x_blocks.shape)  # [2, 512, 4, 4, 4, 16]


Input shape: torch.Size([2, 3, 128, 128, 128])
Patch projection shape: torch.Size([2, 128, 8, 8, 8])
Blockify shape: torch.Size([2, 2048, 2, 4, 1, 4])


In [ ]:
def tokenize_volumetric_data(data, num_bins):
    data_min = data.min()
    data_max = data.max()
    bins = torch.linspace(data_min, data_max, num_bins + 1)
    tokens = torch.bucketize(data, bins)
    return tokens

i = torch.randn(2, 1, 128, 128, 128)
tokens = tokenize_volumetric_data(i, 16)
print(tokens.shape)

torch.Size([2, 1, 128, 128, 128])


In [ ]:
i = torch.randn(2, 128 * 128 * 128)

In [ ]:
i = torch.randn(2, 512, 16, 16, 16)
i = i.view(2, 512, -1)
proj = nn.Linear(4096, 4096)
i = proj(i)
print(i.shape)



NameError: name 'torch' is not defined

In [ ]:
i.reshape(2, 512, 16, 16, 16).shape

torch.Size([2, 512, 16, 16, 16])

In [ ]:
i = torch.randn(2, 256, 64, 64, 64)
rearrange(i, 'b c (h p1) (w p2) (d p3) -> b (h w d) (p1 p2 p3 c)', p1=8, p2=8, p3=8).shape

torch.Size([2, 512, 131072])

In [ ]:
torch.rand(2, 512, 131072).view(2, 512, 16, 16, 16).shape

SyntaxError: invalid syntax (1385936558.py, line 1)

In [ ]:
i.view(2, )

tensor([[[[[[ 1.7017e-01, -1.5988e+00,  8.0365e-01,  ..., -2.8526e-01,
             -5.9000e-01, -4.3778e-01],
            [-1.0419e+00, -4.7698e-01, -4.4592e-01,  ..., -8.8297e-01,
             -7.6208e-01,  4.0584e-01],
            [-1.6355e-01, -1.2900e+00,  4.5960e-01,  ..., -4.4983e-01,
             -6.3738e-01, -2.0550e-01],
            ...,
            [-1.8838e-01, -1.2670e+00,  4.3400e-01,  ..., -4.6208e-01,
             -6.4091e-01, -1.8822e-01],
            [-8.6916e-01, -6.3686e-01, -2.6785e-01,  ..., -7.9779e-01,
             -7.3756e-01,  2.8562e-01],
            [-3.9896e-01, -1.0721e+00,  2.1691e-01,  ..., -5.6592e-01,
             -6.7080e-01, -4.1653e-02]],

           [[-5.8980e-01, -8.9543e-01,  2.0158e-02,  ..., -6.6003e-01,
             -6.9790e-01,  9.1176e-02],
            [-1.6565e+00,  9.1872e-02, -1.0795e+00,  ..., -1.1861e+00,
             -8.4934e-01,  8.3361e-01],
            [-3.3595e-01, -1.1304e+00,  2.8186e-01,  ..., -5.3485e-01,
             -6.6186e-

In [ ]:
i = torch.randn(2, 512, 16, 16, 16, 1)
# Project and preserve blocks 
# proj = nn.Linear(8**3, 256)
# proj(i.view(2, -1, 8**3)).shape
# Back to blocks

In [ ]:
i.shape, i.nelement()

(torch.Size([2, 512, 16, 16, 16, 1]), 4194304)

In [ ]:
def blockify(x, block_size: int):
    """image to blocks
    Args:
        x (Tensor): with shape (B, D, H, W, C)
        block_size (int): edge length of a single square block in units of D, H, W
    """
    b, d, h, w, c = x.shape

    grid_depth = d // block_size
    grid_height = h // block_size
    grid_width = w // block_size
    x = x.reshape(b, grid_depth, block_size, grid_height, block_size, grid_width, block_size, c)

    x = x.permute(0, 1, 3, 5, 2, 4, 6, 7).reshape(
        b, grid_depth * grid_height * grid_width, -1, c
    )  # shape [2, 512, 27, 128]

    return x  # (B, T, N, C)

In [ ]:
blockify(torch.randn(2, 128, 128, 128, 128), 16).shape

torch.Size([2, 512, 4096, 128])

In [ ]:
def deblockify(x, block_size: int):
    """blocks to image
    Args:
        x (Tensor): with shape (B, T, N, C) where T is number of blocks and N is sequence size per block
        block_size (int): edge length of a single square block in units of desired D, H, W
    """
    b, t, _, c = x.shape
    grid_size = round(math.pow(t, 1 / 3))
    depth = height = width = grid_size * block_size
    x = x.reshape(b, grid_size, grid_size, grid_size, block_size, block_size, block_size, c)

    x = x.permute(0, 1, 4, 2, 5, 3, 6, 7).reshape(b, depth, height, width, c)

    return x  # (B, D, H, W, C)

deblockify(torch.randn(2, 512, 4096, 128), 16).shape

torch.Size([2, 128, 128, 128, 128])

In [ ]:
j = i.flatten(2).unsqueeze(-1)
proj = nn.Linear(1, 16)
embedding_tokens = proj(j)
embedding_tokens.shape

torch.Size([2, 512, 4096, 16])

In [ ]:
transformer = TransformerLayer(16, 8, 4)
transformer(embedding_tokens).shape

RuntimeError: [enforce fail at alloc_cpu.cpp:75] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 281474976710656 bytes. Error code 12 (Cannot allocate memory)

In [ ]:
def blockify(tensor, num_blocks):
    batch_size, num_patches, num_elements, embed_dim = tensor.shape
    assert num_elements % num_blocks == 0, "Number of elements must be divisible by the number of blocks."
    
    block_tensor = tensor.reshape(batch_size, num_patches, num_blocks, num_elements // num_blocks, embed_dim)
    return block_tensor

blockify(embedding_tokens, 8).shape

torch.Size([2, 512, 8, 512, 256])

In [ ]:
proj = nn.Linear(16**3, 256)
proj(i.flatten(2)).shape

torch.Size([2, 512, 256])

In [ ]:
embedding_blocks = blockify(embedding_tokens, 8)
embedding_blocks.shape

torch.Size([2, 512, 8, 512, 256])

In [ ]:
embedding_blocks.view

torch.Size([8192, 8, 8, 8, 256])

In [ ]:
transformer = nn.TransformerEncoderLayer(d_model=256, nhead=8)
transformer(embedding_tokens)

AssertionError: query should be unbatched 2D or batched 3D tensor but received 4-D query tensor

In [ ]:
blockify(i, 8).shape

ValueError: too many values to unpack (expected 3)

In [ ]:
_.shape

torch.Size([2, 4096, 8, 8, 8, 256])

In [ ]:
x = torch.randn(2, 512, 8, 16)
x.view(2, )

In [ ]:
x.view(2, 1, 128, 128, 128)

RuntimeError: shape '[2, 1, 128, 128, 128]' is invalid for input of size 131072

In [ ]:
x = torch.randn(2, 128, 8, 256)
B, T, n, C = x.shape
block_size = int(round((n ** (1/3)) * ((T*n)**(1/3))))
blocks_per_dim = int(np.ceil(T ** (1/3)))
H = W = D = block_size * blocks_per_dim
print(H, W, D)
print(block_size, blocks_per_dim)
x = x.reshape(B, C, H, W, D)
print(x.shape)

120 120 120
20 6


RuntimeError: shape '[2, 256, 120, 120, 120]' is invalid for input of size 524288

In [ ]:
def deblockify(x, block_size: int):
    """blocks to image
    Args:
        x (Tensor): with shape (B, T, N, C) where T is number of blocks and N is sequence size per block
        block_size (int): edge length of a single square block in units of desired D, H, W
    """
    b, t, _, c = x.shape
    grid_size = round(math.pow(t, 1 / 3))
    depth = height = width = grid_size * block_size
    x = x.reshape(b, grid_size, grid_size, grid_size, block_size, block_size, block_size, c)

    x = x.permute(0, 1, 4, 2, 5, 3, 6, 7).reshape(b, depth, height, width, c)

    return x  # (B, D, H, W, C)

deblockify(torch.randn(2, 8, 512, 128), 8).shape

torch.Size([2, 16, 16, 16, 128])

In [ ]:
def get_unblocked_shape(blockified_shape, block_size, factor):
    B, T, n, C = blockified_shape
    H, W, D = block_size * factor, block_size * factor, block_size * factor
    H_new = W_new = D_new = H * factor
    T_new = (H_new * W_new * D_new) // (H * W * D)
    n_new = n // T_new
    return [B, C, H_new, W_new, D_new, n_new]

get_unblocked_shape(x.shape, patch_size, 2)

ValueError: too many values to unpack (expected 4)

In [ ]:
Model(patch_size, embed_dim=256, num_hierarchies=1)(x).shape

2) Patch Extraction:  torch.Size([2, 8, 8, 8, 8, 128])
3) Patch Projection:  torch.Size([2, 1024, 256])
Sample patch embedding tensor([-0.1244, -0.6442,  0.1822,  0.1962, -0.6518], grad_fn=<SliceBackward0>)
4) Blockified patch tokens:  torch.Size([2, 128, 8, 256])
5) Transformer output:  torch.Size([2, 128, 8, 256])


RuntimeError: shape '[2, 256, 10, 10, 10]' is invalid for input of size 524288

In [ ]:
x.view(2, 512, 8, 8, 8)

RuntimeError: shape '[2, 512, 8, 8, 8, 1]' is invalid for input of size 1048576

In [ ]:
x.view(BATCH_SIZE, -1, 1024).shape

torch.Size([2, 4096, 1024])

In [ ]:
import torch

def block_aggregation(unblocked_tensor, num_blocks):
    B, C, H, W, D = unblocked_tensor.shape
    block_H, block_W, block_D = num_blocks

    aggregated = unblocked_tensor.reshape(B, C, block_H, H // block_H, block_W, W // block_W, block_D, D // block_D)
    aggregated = aggregated.permute(0, 1, 2, 4, 6, 3, 5, 7)
    aggregated = aggregated.reshape(B, C, H, W, D)
    
    return aggregated

unblocked_tensor = torch.randn(2, 1024, 8, 8, 8)  # Example unblocked tensor
num_blocks = (4, 4, 4)  # Number of blocks in H, W, D dimensions
aggregated_tensor = block_aggregation(unblocked_tensor, num_blocks)

print("Aggregated Tensor Shape:", aggregated_tensor.shape)


Aggregated Tensor Shape: torch.Size([2, 1024, 8, 8, 8])


In [ ]:
_[0].reshape(2, 16, 16, 16, 1024)

tensor([[[[[-1.9976e-01,  2.1819e-01,  4.8646e-01,  ..., -2.4004e-01,
             1.7594e-01,  3.1036e-01],
           [-6.2148e-01, -7.6290e-01,  8.9873e-01,  ..., -1.6078e+00,
             3.6476e-01, -1.3026e+00],
           [ 5.2969e-01, -8.2783e-01,  1.4693e+00,  ...,  4.2239e-01,
             6.2800e-01, -1.6858e-01],
           ...,
           [ 2.5527e-01, -1.8669e+00, -1.5786e-01,  ...,  3.9052e-01,
            -1.0577e+00, -1.3840e+00],
           [-2.1418e+00,  1.4247e+00, -8.3458e-01,  ..., -1.7948e-01,
            -1.0412e-01, -1.1692e+00],
           [ 1.0597e+00,  1.2215e-01,  1.4969e-01,  ...,  1.0156e+00,
            -8.5241e-01, -2.9365e-01]],

          [[ 1.3853e+00, -4.1149e-01,  2.1647e-01,  ...,  1.1280e-01,
             7.0488e-01,  6.2220e-01],
           [ 2.5594e-01, -9.0157e-01, -8.0675e-01,  ..., -8.3670e-01,
            -9.1196e-01, -8.0036e-01],
           [-1.0037e+00, -1.2603e+00,  2.3830e-01,  ...,  1.6137e-01,
            -7.8856e-01,  2.0416e-01],
 

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, kernel_size, stride, padding):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv3d(in_channels, hidden_channels, kernel_size, stride, padding),
            nn.LayerNorm([hidden_channels, 16, 16, 16]),
            nn.ReLU(inplace=True),
            nn.Conv3d(hidden_channels, out_channels, kernel_size, stride, padding),
            nn.LayerNorm([out_channels, 16, 16, 16]),
            nn.ReLU(inplace=True),
        )

        self.relu = nn.ReLU(inplace=True)

        if in_channels != out_channels:
            self.conv_skip = nn.Conv3d(in_channels, out_channels, kernel_size=1, stride=1, padding=0)
        else:
            self.conv_skip = None

    def forward(self, x):
        identity = x
        out = self.model(x)

        if self.conv_skip is not None:
            identity = self.conv_skip(identity)

        out += identity
        out = self.relu(out)

        return out


In [ ]:
x_ = unblock(x)
x_.shape

torch.Size([2, 16, 16, 16, 1024])

In [ ]:
res = ResBlock(1024, 512, 1024, 3, 1, 1)
x_ = x_.reshape(2, 1024, 16, 16, 16)
x_ = res(x_)
x_.shape

torch.Size([2, 1024, 16, 16, 16])

In [ ]:
x_

tensor([[[[[0.0000e+00, 1.2148e+00, 1.2073e+00,  ..., 4.2443e-01,
            3.4750e-01, 1.1113e-01],
           [1.5748e+00, 1.0161e-01, 0.0000e+00,  ..., 1.8440e+00,
            1.9224e+00, 2.8873e-02],
           [0.0000e+00, 0.0000e+00, 6.2818e-01,  ..., 4.4403e-01,
            1.0404e+00, 5.0882e-01],
           ...,
           [1.0571e-02, 7.5891e-01, 0.0000e+00,  ..., 6.4022e-01,
            0.0000e+00, 0.0000e+00],
           [1.0613e+00, 8.2214e-01, 0.0000e+00,  ..., 9.1873e-02,
            0.0000e+00, 0.0000e+00],
           [8.2588e-01, 1.0476e+00, 0.0000e+00,  ..., 1.1294e+00,
            8.8937e-02, 1.2899e-01]],

          [[6.4457e-01, 2.7771e-01, 2.2338e+00,  ..., 1.3932e-01,
            3.3611e-01, 1.1007e+00],
           [7.3797e-01, 1.7525e+00, 1.6491e+00,  ..., 0.0000e+00,
            1.0952e+00, 2.8574e-01],
           [0.0000e+00, 0.0000e+00, 1.4954e+00,  ..., 6.8751e-02,
            0.0000e+00, 5.2194e-01],
           ...,
           [0.0000e+00, 9.8717e-01, 0.0

In [ ]:
i = torch.randn(2, 512, 8, 1024)
i = i.view(2, 1024, 16, 16, 16)
res = ResBlock(1024, 512, 1024, 3, 1, 1)
res(i).shape

torch.Size([2, 1024, 16, 16, 16])

In [ ]:
[i.shape for i in x]

[torch.Size([2, 512, 8, 1024]),
 torch.Size([2, 256, 8, 1024]),
 torch.Size([2, 128, 8, 1024]),
 torch.Size([2, 64, 8, 1024]),
 torch.Size([2, 32, 8, 1024]),
 torch.Size([2, 16, 8, 1024]),
 torch.Size([2, 8, 8, 1024])]

In [ ]:
for i in x:
    i =
    res = ResBlock(1024, 512, 1024, 3, 1, 1)(i)
    print(res.shape)

RuntimeError: Given groups=1, weight of size [512, 1024, 3, 3, 3], expected input[1, 2, 512, 8, 1024] to have 1024 channels, but got 2 channels instead

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm1 = nn.InstanceNorm3d(out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1)
        self.norm2 = nn.InstanceNorm3d(out_channels)

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.norm1(out)
        out = F.relu(out)
        out = self.conv2(out)
        out = self.norm2(out)
        out += residual
        out = F.relu(out)
        return out


encoded_outputs = x
# 1. Initial bottleneck
C = 128
bottleneck = nn.Conv3d(1024, C, kernel_size=3, padding=1)(encoded_outputs[-1])

# 2. Upsampling and concatenation
for encoded_output in reversed(encoded_outputs[:-1]):
    # a. Upsample
    upsample = nn.ConvTranspose3d(C, C, kernel_size=3, stride=2, padding=1, output_padding=1)
    bottleneck = upsample(bottleneck)

    # b. Concatenate
    concatenated = torch.cat((bottleneck, encoded_output), dim=1)

    # c. Residual block
    res_block = ResidualBlock(2 * C, C)
    bottleneck = res_block(concatenated)

# 3. Final upsampling
final_upsample = nn.ConvTranspose3d(C, C, kernel_size=3, stride=2, padding=1, output_padding=1)
upsampled_output = final_upsample(bottleneck)

# 4. Segmentation mask
num_classes = 256  # Assuming 256 block types in Minecraft schematics
seg_mask_layer = nn.Conv3d(C, num_classes, kernel_size=1)
segmentation_mask = seg_mask_layer(upsampled_output)

# 5. Reshape and apply softmax
softmax = nn.Softmax(dim=1)
segmentation_mask = softmax(segmentation_mask)
generated_schematics = segmentation_mask.view(2, 1, 128, 128, 128, num_classes)


RuntimeError: Given groups=1, weight of size [128, 1024, 3, 3, 3], expected input[1, 2, 1, 8, 1024] to have 1024 channels, but got 2 channels instead

In [ ]:
conv = nn.Conv3d(1, 1, 3, groups=1)
conv(x).shape

RuntimeError: Given groups=1, weight of size [1, 1, 3, 3, 3], expected input[1, 2, 1, 8, 1024] to have 1 channels, but got 2 channels instead

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerLayer(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio):
        super().__init__()
        self.attention = nn.MultiheadAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * mlp_ratio),
            nn.ReLU(),
            nn.Linear(embed_dim * mlp_ratio, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        B, num_blocks, block_size, emb_dim = x.shape
        x = x.permute(1, 2, 0, 3).reshape(num_blocks * block_size, B, emb_dim)
        
        attn_output, _ = self.attention(x, x, x)
        x = x + attn_output
        x = self.norm1(x)

        mlp_output = self.mlp(x)
        x = x + mlp_output
        x = self.norm2(x)

        x = x.reshape(num_blocks, block_size, B, emb_dim).permute(2, 0, 1, 3)
        return x

# Instantiate a TransformerLayer and apply it to the blockified tensor
embed_dim = 1024
num_heads = 8
mlp_ratio = 4
transformer_layer = TransformerLayer(embed_dim, num_heads, mlp_ratio)

print(blockified[:, 0, 0, :5])
transformed = transformer_layer(blockified)
print("Transformed tensor shape: ", transformed.shape)
print(transformed[:, 0, 0, :5])

tensor([[-1.5859, -1.4756, -0.4196,  1.1843, -0.2850],
        [-0.1837,  1.5177, -0.5694, -1.5482, -0.6980]])
Transformed tensor shape:  torch.Size([2, 32, 16, 1024])
tensor([[-1.7016, -1.2726, -0.1917,  1.0940, -0.6495],
        [ 0.0187,  1.6327, -0.3248, -1.1757, -1.2742]],
       grad_fn=<SliceBackward0>)


In [ ]:
class HierarchicalTransformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio, num_layers, num_hierarchies):
        super().__init__()
        self.num_hierarchies = num_hierarchies
        self.hierarchies = nn.ModuleList([
            nn.ModuleList([
                TransformerLayer(embed_dim, num_heads, mlp_ratio)
                for _ in range(num_layers)
            ])
            for _ in range(num_hierarchies)
        ])

    def forward(self, x):
        B, num_blocks, block_size, emb_dim = x.shape
        for hierarchy in self.hierarchies:
            for layer in hierarchy:
                x = layer(x)
            x = block_aggregation(x)  # Assuming block_aggregation is implemented
            x = downsample(x)  # Assuming downsample is implemented
        return x

# Create a HierarchicalTransformerEncoder instance
embed_dim = 1024
num_heads = 8
mlp_ratio = 4
num_layers = 2
num_hierarchies = 3

encoder = HierarchicalTransformerEncoder(embed_dim, num_heads, mlp_ratio, num_layers, num_hierarchies)

# Apply the Hierarchical Transformer Encoder to the blockified tensor
encoded = encoder(blockified)
print("5) Hierarchical Transformer Encoder: ", encoded.shape)


TypeError: layer_norm(): argument 'input' (position 1) must be Tensor, not int

In [ ]:
def blockify(patch_projected, block_size):
    B, num_patches, emb_dim = patch_projected.shape
    num_blocks = num_patches // block_size
    blockified = patch_projected.view(B, num_blocks, block_size, emb_dim)
    return blockified

# Given patch_projected of shape (2, 512, 1024)
patch_projected = torch.randn(2, 512, 1024)

# Perform blockify
block_size = 16
blockified = blockify(patch_projected, block_size)

print("4) Blockify: ", blockified.shape)


4) Blockify:  torch.Size([2, 32, 16, 1024])
